# Notebook 07 — Modeling Design & Validation Framework

**Project:** WR-MovementIQ  
**Purpose:** Freeze the modeling population, construct leakage-safe player-grouped validation folds, and export reproducible fold assignments.

This notebook does not claim that any model has predictive value. It prepares and audits the validation design before model fitting.

## Research design

- Primary modeling population: `wr_analysis_dataset_core_common.parquet`
- Expected sample: 93 player-seasons from 54 unique receivers
- Outcomes: six pre-specified game movement outcomes
- Predictor groups:
  - Traditional: `forty`, `vertical`, `broad_jump`
  - Tracking: six Combine tracking-derived movement features
  - Combined: all nine predictors
- Outer validation:
  - Five-fold `GroupKFold`, grouping by `nfl_id`
  - Leave-one-draft-cohort-out (LOCO), holding out each `draft_year`
- No random row-level splitting.
- No imputation or feature selection outside training folds.

**Important:** The earlier 116-player-season result remains an unresolved historical discrepancy. This notebook uses the audited 93-row common comparison sample and does not force the data to match the historical count.

## 1. Environment and project paths

Run this notebook from the project root (`D:\PROJECTS\WR-MovementIQ`) or update `PROJECT_ROOT` if necessary.

In [1]:
from pathlib import Path
import json
import hashlib
import platform
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupKFold

PROJECT_ROOT = Path.cwd()

# If launched from the notebooks directory, move to its parent.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FEATURES = PROJECT_ROOT / "data" / "features"
REPORT_DIR = PROJECT_ROOT / "reports" / "modeling_design"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CORE_DATA_PATH = DATA_FEATURES / "wr_analysis_dataset_core_common.parquet"
PRIMARY_DATA_PATH = DATA_FEATURES / "wr_analysis_dataset_primary.parquet"

assert CORE_DATA_PATH.exists(), f"Missing file: {CORE_DATA_PATH}"
assert PRIMARY_DATA_PATH.exists(), f"Missing file: {PRIMARY_DATA_PATH}"

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Core dataset:", CORE_DATA_PATH)
print("Report directory:", REPORT_DIR)

Project root: D:\PROJECTS\WR-MovementIQ
Python: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.22621-SP0
Pandas: 2.2.2
Scikit-learn: 1.5.0
Core dataset: D:\PROJECTS\WR-MovementIQ\data\features\wr_analysis_dataset_core_common.parquet
Report directory: D:\PROJECTS\WR-MovementIQ\reports\modeling_design


## 2. Load frozen datasets and verify keys

The core comparison sample is the common complete-case sample for all six outcomes, all three traditional predictors, and all six tracking predictors.

In [2]:
core = pd.read_parquet(CORE_DATA_PATH)
primary = pd.read_parquet(PRIMARY_DATA_PATH)

GAME_KEY = ["draft_year", "nfl_id", "season"]

OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_p95_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_peak_acceleration_raw_median",
    "game_path_efficiency_median",
]

TRADITIONAL = ["forty", "vertical", "broad_jump"]

TRACKING = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_p95_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_peak_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
]

MODEL_FEATURES = {
    "TRADITIONAL": TRADITIONAL,
    "TRACKING": TRACKING,
    "COMBINED": TRADITIONAL + TRACKING,
}

required = GAME_KEY + OUTCOMES + TRADITIONAL + TRACKING
missing_columns = sorted(set(required) - set(core.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"

assert not core[GAME_KEY].isna().any().any(), "Null primary-key values found."
assert not core.duplicated(GAME_KEY).any(), "Duplicate player-season keys found."
assert core[required].notna().all().all(), "Missing values found in core modeling columns."

print("Core shape:", core.shape)
print("Unique receivers:", core["nfl_id"].nunique())
print("Draft cohorts:", sorted(core["draft_year"].unique().tolist()))
print("Primary sample shape:", primary.shape)
display(core.groupby("draft_year").agg(
    player_seasons=("nfl_id", "size"),
    unique_players=("nfl_id", "nunique")
).reset_index())

Core shape: (93, 103)
Unique receivers: 54
Draft cohorts: [2023, 2024, 2025]
Primary sample shape: (114, 103)


,draft_year,player_seasons,unique_players
0,2023,45,20
1,2024,34,20
2,2025,14,14


## 3. Freeze sample and model specifications

The expected values below are assertions against the audited dataset. If the local file differs, stop and investigate rather than silently changing the study population.

In [3]:
EXPECTED_ROWS = 93
EXPECTED_PLAYERS = 54
EXPECTED_COHORTS = {2023, 2024, 2025}

assert len(core) == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS} rows; found {len(core)}. "
    "Check the frozen dataset before proceeding."
)
assert core["nfl_id"].nunique() == EXPECTED_PLAYERS, (
    f"Expected {EXPECTED_PLAYERS} players; found {core['nfl_id'].nunique()}."
)
assert set(core["draft_year"].unique()) == EXPECTED_COHORTS

specification_rows = []
for model_name, features in MODEL_FEATURES.items():
    specification_rows.append({
        "model": model_name,
        "predictor_count": len(features),
        "predictors": ", ".join(features)
    })

specification_df = pd.DataFrame(specification_rows)
display(specification_df)

# Persist a hash so later modeling notebooks can verify the same input file.
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

core_sha256 = sha256_file(CORE_DATA_PATH)
print("Core dataset SHA256:", core_sha256)

,model,predictor_count,predictors
0,TRADITIONAL,3,"forty, vertical, broad_jump"
1,TRACKING,6,"combine_mean_speed_median_across_drills, combi..."
2,COMBINED,9,"forty, vertical, broad_jump, combine_mean_spee..."


Core dataset SHA256: 02fbd802522973f7c4d100da57cd680f7bbfde6346a0431a7aa1fe5e25807c5c


## 4. Five-fold player-grouped outer validation

`GroupKFold` ensures that every receiver belongs to exactly one outer test fold. A receiver with multiple seasons cannot appear in both training and test data.

We use the same outer fold assignment for every outcome and model specification.

In [4]:
N_OUTER_SPLITS = 5
groups = core["nfl_id"].to_numpy()

outer_cv = GroupKFold(n_splits=N_OUTER_SPLITS)

outer_assignments = np.full(len(core), -1, dtype=int)
outer_fold_rows = []

for fold_id, (train_idx, test_idx) in enumerate(
    outer_cv.split(core, groups=groups), start=1
):
    train_players = set(core.iloc[train_idx]["nfl_id"])
    test_players = set(core.iloc[test_idx]["nfl_id"])

    assert train_players.isdisjoint(test_players), (
        f"Player leakage detected in outer fold {fold_id}"
    )

    outer_assignments[test_idx] = fold_id

    outer_fold_rows.append({
        "fold": fold_id,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_players": len(train_players),
        "test_players": len(test_players),
        "train_draft_years": ",".join(
            map(str, sorted(core.iloc[train_idx]["draft_year"].unique()))
        ),
        "test_draft_years": ",".join(
            map(str, sorted(core.iloc[test_idx]["draft_year"].unique()))
        ),
    })

assert (outer_assignments > 0).all()
core["outer_fold"] = outer_assignments

outer_fold_summary = pd.DataFrame(outer_fold_rows)
display(outer_fold_summary)

# Each player must map to one and only one outer fold.
player_fold_counts = core.groupby("nfl_id")["outer_fold"].nunique()
assert player_fold_counts.max() == 1
print("Players assigned to exactly one outer fold:", len(player_fold_counts))

,fold,train_rows,test_rows,train_players,test_players,train_draft_years,test_draft_years
0,1,74,19,43,11,"2023,2024,2025","2023,2024,2025"
1,2,74,19,43,11,"2023,2024,2025","2023,2024,2025"
2,3,74,19,43,11,"2023,2024,2025","2023,2024,2025"
3,4,75,18,44,10,"2023,2024,2025","2023,2024,2025"
4,5,75,18,43,11,"2023,2024,2025","2023,2024,2025"


Players assigned to exactly one outer fold: 54


## 5. Inner grouped folds for nested hyperparameter selection

Inner folds are generated separately within each outer training partition. These are the folds used for selecting Ridge regularization strength in the later modeling notebook.

The inner split count is capped at four and checked against the number of unique training receivers.

In [5]:
inner_fold_rows = []

for outer_fold in range(1, N_OUTER_SPLITS + 1):
    outer_train = core.loc[core["outer_fold"] != outer_fold].copy()
    n_train_players = outer_train["nfl_id"].nunique()
    n_inner_splits = min(4, n_train_players)

    assert n_inner_splits >= 2

    inner_cv = GroupKFold(n_splits=n_inner_splits)
    inner_groups = outer_train["nfl_id"].to_numpy()

    for inner_fold, (itr, iva) in enumerate(
        inner_cv.split(outer_train, groups=inner_groups), start=1
    ):
        train_ids = set(outer_train.iloc[itr]["nfl_id"])
        valid_ids = set(outer_train.iloc[iva]["nfl_id"])

        assert train_ids.isdisjoint(valid_ids)

        inner_fold_rows.append({
            "outer_fold": outer_fold,
            "inner_fold": inner_fold,
            "inner_train_rows": len(itr),
            "inner_validation_rows": len(iva),
            "inner_train_players": len(train_ids),
            "inner_validation_players": len(valid_ids),
        })

inner_fold_summary = pd.DataFrame(inner_fold_rows)
display(inner_fold_summary)

,outer_fold,inner_fold,inner_train_rows,inner_validation_rows,inner_train_players,inner_validation_players
0,1,1,55,19,32,11
1,1,2,55,19,32,11
2,1,3,56,18,33,10
3,1,4,56,18,32,11
4,2,1,55,19,32,11
5,2,2,55,19,32,11
6,2,3,56,18,33,10
7,2,4,56,18,32,11
8,3,1,55,19,32,11
9,3,2,55,19,32,11


## 6. Leave-one-draft-cohort-out (LOCO) validation

Each draft cohort is held out in turn. All player-seasons belonging to the held-out draft cohort are excluded from training.

This tests transfer to an unseen draft cohort. It is a different and more demanding validation question than grouped random folds.

In [6]:
loco_rows = []

for held_out_year in sorted(core["draft_year"].unique()):
    train = core.loc[core["draft_year"] != held_out_year]
    test = core.loc[core["draft_year"] == held_out_year]

    train_players = set(train["nfl_id"])
    test_players = set(test["nfl_id"])

    # A player should not belong to multiple draft cohorts.
    assert train_players.isdisjoint(test_players), (
        f"Player overlap across LOCO split for cohort {held_out_year}"
    )

    loco_rows.append({
        "held_out_draft_year": int(held_out_year),
        "train_rows": len(train),
        "test_rows": len(test),
        "train_players": len(train_players),
        "test_players": len(test_players),
        "train_cohorts": ",".join(
            map(str, sorted(train["draft_year"].unique()))
        ),
    })

loco_summary = pd.DataFrame(loco_rows)
display(loco_summary)

,held_out_draft_year,train_rows,test_rows,train_players,test_players,train_cohorts
0,2023,48,45,34,20,"2024,2025"
1,2024,59,34,34,20,"2023,2025"
2,2025,79,14,40,14,"2023,2024"


## 7. Outcome-specific sample audit

The frozen core dataset has complete values for all six outcomes and all nine core predictors. We still export an outcome-specific audit so downstream notebooks can verify that the exact sample used for each model is consistent.

In [7]:
outcome_sample_rows = []

for outcome in OUTCOMES:
    for model_name, predictors in MODEL_FEATURES.items():
        required_cols = [outcome] + predictors
        mask = core[required_cols].notna().all(axis=1)
        subset = core.loc[mask]

        outcome_sample_rows.append({
            "outcome": outcome,
            "model": model_name,
            "rows": len(subset),
            "players": subset["nfl_id"].nunique(),
            "draft_cohorts": subset["draft_year"].nunique(),
        })

outcome_sample_audit = pd.DataFrame(outcome_sample_rows)
display(outcome_sample_audit)

assert outcome_sample_audit["rows"].nunique() == 1
assert outcome_sample_audit["players"].nunique() == 1

,outcome,model,rows,players,draft_cohorts
0,game_mean_speed_raw_median,TRADITIONAL,93,54,3
1,game_mean_speed_raw_median,TRACKING,93,54,3
2,game_mean_speed_raw_median,COMBINED,93,54,3
3,game_peak_speed_raw_median,TRADITIONAL,93,54,3
4,game_peak_speed_raw_median,TRACKING,93,54,3
5,game_peak_speed_raw_median,COMBINED,93,54,3
6,game_p95_speed_raw_median,TRADITIONAL,93,54,3
7,game_p95_speed_raw_median,TRACKING,93,54,3
8,game_p95_speed_raw_median,COMBINED,93,54,3
9,game_mean_acceleration_raw_median,TRADITIONAL,93,54,3


## 8. Leakage and integrity assertions

These checks are mandatory. Any failure should stop model fitting.

In [8]:
# Global key integrity
assert not core.duplicated(GAME_KEY).any()
assert core["nfl_id"].notna().all()

# Fold integrity
assert core["outer_fold"].between(1, N_OUTER_SPLITS).all()
assert core.groupby("nfl_id")["outer_fold"].nunique().max() == 1

# No overlap between outer train and test players.
for fold in range(1, N_OUTER_SPLITS + 1):
    train_ids = set(core.loc[core["outer_fold"] != fold, "nfl_id"])
    test_ids = set(core.loc[core["outer_fold"] == fold, "nfl_id"])
    assert train_ids.isdisjoint(test_ids)

# All feature values in the core sample are present and numeric.
for col in OUTCOMES + TRADITIONAL + TRACKING:
    core[col] = pd.to_numeric(core[col], errors="raise")
    assert np.isfinite(core[col].to_numpy(dtype=float)).all(), (
        f"Non-finite values found in {col}"
    )

print("PASS: keys, completeness, numeric values, and player-level fold isolation.")

PASS: keys, completeness, numeric values, and player-level fold isolation.


## 9. Export reproducible validation artifacts

Exports include:
- Player-season rows with outer fold assignment.
- Outer fold summary.
- Inner fold summary.
- LOCO cohort summary.
- Outcome/model sample audit.
- Modeling specification and dataset fingerprint manifest.

These artifacts are inputs to Notebook 08. Do not manually edit fold assignments.

In [9]:
# Export row-level fold assignment, retaining only required identifiers and fold.
fold_assignment = core[
    GAME_KEY + ["outer_fold"]
].sort_values(GAME_KEY).reset_index(drop=True)

fold_assignment.to_csv(
    REPORT_DIR / "outer_grouped_fold_assignments.csv", index=False
)
fold_assignment.to_parquet(
    REPORT_DIR / "outer_grouped_fold_assignments.parquet", index=False
)

outer_fold_summary.to_csv(
    REPORT_DIR / "outer_fold_summary.csv", index=False
)
inner_fold_summary.to_csv(
    REPORT_DIR / "inner_fold_summary.csv", index=False
)
loco_summary.to_csv(
    REPORT_DIR / "loco_cohort_summary.csv", index=False
)
outcome_sample_audit.to_csv(
    REPORT_DIR / "outcome_model_sample_audit.csv", index=False
)
specification_df.to_csv(
    REPORT_DIR / "model_specifications.csv", index=False
)

manifest = {
    "notebook": "07_modeling_design_validation",
    "core_dataset": str(CORE_DATA_PATH.relative_to(PROJECT_ROOT)),
    "core_dataset_sha256": core_sha256,
    "rows": int(len(core)),
    "unique_players": int(core["nfl_id"].nunique()),
    "draft_years": sorted(map(int, core["draft_year"].unique())),
    "primary_key": GAME_KEY,
    "outcomes": OUTCOMES,
    "model_features": MODEL_FEATURES,
    "outer_validation": {
        "method": "GroupKFold",
        "n_splits": N_OUTER_SPLITS,
        "group": "nfl_id",
    },
    "inner_validation": {
        "method": "GroupKFold",
        "max_splits": 4,
        "group": "nfl_id",
    },
    "generalization_validation": "Leave-one-draft-cohort-out",
    "imputation": False,
    "preprocessing_policy": (
        "Fit all preprocessing within each training fold in a sklearn Pipeline."
    ),
    "historical_116_row_discrepancy": (
        "Unresolved; not used to alter the audited 93-row core sample."
    ),
}

with open(REPORT_DIR / "modeling_design_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("EXPORTED ARTIFACTS")
for path in sorted(REPORT_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name:45s} {path.stat().st_size:>10,d} bytes")

print("\nNOTEBOOK 07 VALIDATION DESIGN: PASS")

EXPORTED ARTIFACTS
inner_fold_summary.csv                               447 bytes
loco_cohort_summary.csv                              173 bytes
model_specifications.csv                             662 bytes
modeling_design_manifest.json                      1,995 bytes
outcome_model_sample_audit.csv                       912 bytes
outer_fold_summary.csv                               334 bytes
outer_grouped_fold_assignments.csv                 1,804 bytes
outer_grouped_fold_assignments.parquet             3,594 bytes

NOTEBOOK 07 VALIDATION DESIGN: PASS


## 10. Interpretation and handoff

A successful run confirms that the validation design is structurally sound. It does **not** establish that any model predicts well.

Before fitting models in Notebook 08:
1. Preserve this notebook and its exported fold assignments.
2. Use `outer_fold` as the fixed outer split for every outcome and model.
3. Tune Ridge alpha only within each outer training partition using the inner player-grouped folds.
4. Fit scaling and any other learned preprocessing inside the training pipeline.
5. Compare models on paired outer-fold predictions from identical observations.
6. Report both player-season and player-level results, explaining how repeated seasons are aggregated.
7. Treat LOCO as a separate cohort-transfer analysis, not as interchangeable with GroupKFold.

**Next notebook:** Notebook 08 — Nested Ridge Modeling and Out-of-Fold Evaluation.